In [ ]:
# 安裝套件
%pip install --upgrade pip
%pip install -r requirements.txt

In [ ]:
# 錄製影片素材
import cv2
import os
import time
import shutil
import subprocess
from datetime import datetime
from dotenv import load_dotenv
from IPython.display import clear_output, display, Image

load_dotenv()

os.environ["OPENCV_FFMPEG_CAPTURE_OPTIONS"] = "timeout;10000000|rw_timeout;10000000"

# 參數設定
CCTV_URL = os.environ["CCTV_URL"]
RECORD_SECONDS = 3600        # 真實牆鐘秒數（要錄 1 小時就填 3600）
WRITER_FPS = 30.0            # 暫存檔的名目 fps，數值不重要（稍後依真實 fps 重新標記）
PREVIEW_EVERY_SEC = 5.0      # 預覽更新間隔（牆鐘秒），降頻以減少對擷取速率的拖累

# 儲存路徑
save_path = "origin.mp4"
temp_path = "temp_origin.mp4"


def remux_to_real_fps(src, dst, fps_out, width, height):
    """把暫存檔以 fps_out 重新標記為 dst。
    有 ffmpeg 時走 -c copy（不重新編碼、快、無畫質損失）；
    沒有 ffmpeg 時退回純 OpenCV 重寫（會重新編碼）。"""
    if shutil.which("ffmpeg") is not None:
        # -r 置於 -i 之前：以 fps_out 重新指定輸入時間戳；-c copy 不重新編碼
        cmd = ["ffmpeg", "-y", "-r", f"{fps_out:.6f}", "-i", src, "-c", "copy", dst]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            return True
        print("⚠️ ffmpeg 重新封裝失敗，改用 OpenCV 重寫。ffmpeg 錯誤：")
        print(result.stderr[-1500:])

    # 退回方案：用 OpenCV 以 fps_out 逐幀重寫（無 ffmpeg 時）
    print("ℹ️ 未偵測到 ffmpeg，改用 OpenCV 以真實 fps 重寫（會重新編碼，較慢）。")
    rcap = cv2.VideoCapture(src)
    rout = cv2.VideoWriter(dst, cv2.VideoWriter_fourcc(*"mp4v"), fps_out, (width, height))
    while True:
        ok, f = rcap.read()
        if not ok:
            break
        rout.write(f)
    rcap.release()
    rout.release()
    return True


# 開啟串流
cap = cv2.VideoCapture(CCTV_URL)

if not cap.isOpened():
    print(f"❌ 無法開啟串流 {CCTV_URL}")
else:
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))  or 1280
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 720
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(temp_path, fourcc, WRITER_FPS, (width, height))

    print(f"🚀 開始錄製純影片...")
    print(f"📂 暫存路徑: {temp_path} -> 最終: {save_path}")
    print(f"⏱️  預計時長: {RECORD_SECONDS} 秒（以真實牆鐘時間為準）")

    frame_count = 0
    start = time.monotonic()       # 以牆鐘時間決定何時結束，確保 1 小時≈1 小時
    last_preview = start

    try:
        while time.monotonic() - start < RECORD_SECONDS:
            ret, frame = cap.read()
            if not ret:
                print("⚠️ 串流暫時中斷，重試中...")
                time.sleep(1)
                continue

            out.write(frame)
            frame_count += 1

            # 預覽：以牆鐘時間降頻（約每 PREVIEW_EVERY_SEC 秒一次）
            now = time.monotonic()
            if now - last_preview >= PREVIEW_EVERY_SEC:
                last_preview = now
                clear_output(wait=True)
                pct = min((now - start) / RECORD_SECONDS * 100, 100.0)
                print(f"🔴 錄製中：{pct:.1f}% [{frame_count} 影格]")
                resize_frame = cv2.resize(frame, (640, 360))
                _, encoded_img = cv2.imencode('.jpg', resize_frame)
                display(Image(data=encoded_img.tobytes()))

    except KeyboardInterrupt:
        print("\n🛑 手動中斷錄製")

    finally:
        elapsed = time.monotonic() - start
        cap.release()
        out.release()

    # 以真實平均 fps（總影格 / 真實秒數）重新標記，校正播放速率與時間戳
    if frame_count == 0 or elapsed <= 0:
        print("⚠️ 未擷取到任何影格，未產生 origin.mp4。請檢查串流來源。")
    else:
        fps_out = frame_count / elapsed
        print(f"📊 擷取 {frame_count} 影格 / 實際 {elapsed:.1f} 秒 -> 真實平均 fps = {fps_out:.3f}")
        if remux_to_real_fps(temp_path, save_path, fps_out, width, height):
            try:
                os.remove(temp_path)
            except OSError:
                pass
            print(f"✅ 錄製結束，存儲於: {os.path.abspath(save_path)}（fps={fps_out:.3f}）")
        else:
            print("❌ 重新封裝失敗，已保留暫存檔以便除錯：", os.path.abspath(temp_path))


In [ ]:
# 逐幀本地推論（YOLO 偵測 + ByteTrack 追蹤），將標註、時間存入 SQLite
import cv2
import os
import json
import sqlite3
from dotenv import load_dotenv
from ultralytics import YOLO

load_dotenv()

# --- 1. 參數設定 ---
INPUT_VIDEO = "origin.mp4"
DB_PATH = "detections.db"
# 本地模型路徑（稍後放入自己的 .pt 即可），與偵測信心門檻
# CONFIDENCE 採 Ultralytics 預設 0.25；過高（如 0.9 以上）會濾掉絕大多數偵測，可依模型表現調整
MODEL_PATH = os.environ.get("MODEL_PATH", "model.pt")
CONFIDENCE = float(os.environ.get("CONFIDENCE", "0.25"))

# --- 2. 初始化 SQLite (每次執行重置) ---
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.executescript("""
DROP TABLE IF EXISTS frames;
CREATE TABLE frames (
    frame_id         INTEGER PRIMARY KEY,
    pts_ms           REAL,
    predictions_json TEXT
);
""")
conn.commit()
print(f"SQLite 已重置：{os.path.abspath(DB_PATH)}")

# --- 3. 載入本地 YOLO 模型（內建 ByteTrack） ---
model = YOLO(MODEL_PATH)
print(f"已載入模型：{MODEL_PATH}（類別：{model.names}）")

# --- 4. 逐幀讀取、追蹤、收集標註 ---
cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    conn.close()
    raise RuntimeError(f"無法開啟原影片：{INPUT_VIDEO}")

fps = cap.get(cv2.CAP_PROP_FPS) or 20.0
frame_buffer = []  # (frame_id, pts_ms, predictions_json)
frame_id = 0
debug_samples_left = 3  # 印出前幾個非空 payload，方便確認欄位

print(f"開始本地推論：{INPUT_VIDEO} @ {fps:.1f}fps，conf={CONFIDENCE}")
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # persist=True 維持跨幀軌跡；tracker 使用 ultralytics 內建 bytetrack.yaml
    results = model.track(
        frame,
        persist=True,
        conf=CONFIDENCE,
        tracker="bytetrack.yaml",
        verbose=False,
    )
    r = results[0]

    # 取出有 tracker_id 的框，整理成每幀的標註清單
    preds = []
    boxes = getattr(r, "boxes", None)
    if boxes is not None and boxes.id is not None:
        xywh = boxes.xywh.cpu().numpy()           # 中心座標 (cx, cy, w, h)
        ids = boxes.id.cpu().numpy().astype(int)
        clss = boxes.cls.cpu().numpy().astype(int)
        confs = boxes.conf.cpu().numpy()
        for (cx, cy, bw, bh), tid, c, cf in zip(xywh, ids, clss, confs):
            preds.append({
                "x": float(cx),
                "y": float(cy),
                "width": float(bw),
                "height": float(bh),
                "class": model.names[int(c)],
                "tracker_id": int(tid),
                "confidence": float(cf),
            })

    pts_ms = (frame_id / fps) * 1000.0
    frame_buffer.append((frame_id, pts_ms, json.dumps(preds)))

    if debug_samples_left > 0 and preds:
        print(f"[payload 樣本] frame {frame_id}: {len(preds)} 筆，第一筆={preds[0]}")
        debug_samples_left -= 1

    frame_id += 1
    if frame_id % 200 == 0:
        print(f"  已處理 {frame_id} 影格…")

cap.release()

# --- 5. 批次寫入 DB ---
cur.executemany("INSERT OR REPLACE INTO frames VALUES (?,?,?)", frame_buffer)
conn.commit()

# 統計含標註的影格數，立即得知是否真的取得 predictions
n_total, n_nonempty = cur.execute(
    "SELECT COUNT(*), SUM(CASE WHEN predictions_json NOT IN ('null','[]') THEN 1 ELSE 0 END) FROM frames"
).fetchone()
conn.close()

print(f"\n=== 步驟一完成 ===")
print(f"總影格 {n_total} 筆，其中含標註 {n_nonempty or 0} 筆 -> {os.path.abspath(DB_PATH)}")
if not n_nonempty:
    print("⚠️ 所有 predictions 皆為空：請確認 MODEL_PATH 指向有效模型，"
          "且 CONFIDENCE 門檻未過高（預設 0.25）。")


In [ ]:
# 從 SQLite 解析標註、計數並重畫到原影片，輸出 labeled.mp4
import cv2
import json
import sqlite3
from collections import defaultdict

INPUT_VIDEO = "origin.mp4"
DB_PATH = "detections.db"
OUTPUT_VIDEO = "labeled.mp4"
COUNT_LINE_Y = 120       # 計數警戒線 y
COUNT_LINE_X_MIN = 0     # 計數線段 x 起點
COUNT_LINE_X_MAX = 160   # 計數線段 x 終點

def parse_predictions(raw):
    """把步驟一原樣保存的 predictions 解析成 list[dict]，容忍多種結構。"""
    if isinstance(raw, str):
        try:
            raw = json.loads(raw)
        except json.JSONDecodeError:
            return []
    if raw is None:
        return []
    if isinstance(raw, dict):
        # 常見：{"predictions": [...]}；否則找出第一個 list 值
        if isinstance(raw.get("predictions"), list):
            return raw["predictions"]
        for v in raw.values():
            if isinstance(v, list):
                return v
        return [raw]  # 單一物件 dict
    if isinstance(raw, list):
        return raw
    return []

# --- 1. 讀取所有影格的原始標註 ---
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

rows = cur.execute(
    "SELECT frame_id, pts_ms, predictions_json FROM frames ORDER BY frame_id"
).fetchall()
preds_by_frame = {fid: (pts, parse_predictions(pj)) for fid, pts, pj in rows}

# 診斷：印出第一個非空樣本的結構，確認欄位名
for fid, (pts, plist) in preds_by_frame.items():
    if plist:
        print(f"[解析樣本] frame {fid}: {len(plist)} 筆，第一筆 keys="
              f"{list(plist[0].keys()) if isinstance(plist[0], dict) else type(plist[0]).__name__}")
        break
else:
    print("⚠️ frames 表中沒有任何非空 predictions，請先檢查步驟一輸出。")

# --- 2. 依 frame_id 順序偵測跨線並累計去重計數，並把計數落地 frame_counts ---
cur.executescript("""
DROP TABLE IF EXISTS frame_counts;
CREATE TABLE frame_counts (
    frame_id    INTEGER PRIMARY KEY,
    pts_ms      REAL,
    counts_json TEXT
);
""")

class_ids = defaultdict(set)  # 各類別已跨線(計數過)的 tracker_id
prev_pos = {}                 # tracker_id -> 上一次出現的中心 (x, y)
count_rows = []
for fid in sorted(preds_by_frame):
    pts, plist = preds_by_frame[fid]
    for pred in plist:
        if not isinstance(pred, dict):
            continue
        track_id = pred.get("tracker_id")
        vtype = pred.get("class")
        x = pred.get("x")
        y = pred.get("y")
        if track_id is None or vtype is None or x is None or y is None:
            continue
        prev = prev_pos.get(track_id)
        prev_pos[track_id] = (x, y)   # 不論是否計數，都更新軌跡上一點
        if prev is None:              # 第一次看到此 track，無法判定跨線
            continue
        if any(track_id in ids for ids in class_ids.values()):
            continue                  # 已計數過
        px, py = prev
        # 前後兩次出現分屬計數線兩側 -> 連線穿過 y=COUNT_LINE_Y（雙向）
        if (py - COUNT_LINE_Y) * (y - COUNT_LINE_Y) < 0:
            # 連線與 y=COUNT_LINE_Y 的交點 x（y != py 已由上式保證）
            t = (COUNT_LINE_Y - py) / (y - py)
            cross_x = px + t * (x - px)
            if COUNT_LINE_X_MIN <= cross_x <= COUNT_LINE_X_MAX:
                class_ids[vtype].add(track_id)
    snapshot = {cls: len(ids) for cls, ids in class_ids.items()}
    count_rows.append((fid, pts, json.dumps(snapshot)))

cur.executemany("INSERT OR REPLACE INTO frame_counts VALUES (?,?,?)", count_rows)
conn.commit()

final_total = sum(len(ids) for ids in class_ids.values())
summary = ", ".join(f"{cls}={len(ids)}" for cls, ids in sorted(class_ids.items()))
print(f"=== 計數完成 === Total={final_total}, {summary}")

# 把每幀累計計數讀回記憶體供繪製
counts_by_frame = {fid: json.loads(cj) for fid, _, cj in count_rows}

# --- 3. 開原影片，逐幀重畫框 + 計數 + 計數線 ---
cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    conn.close()
    raise RuntimeError(f"無法開啟原影片：{INPUT_VIDEO}")

fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
print(f"開始繪製：{INPUT_VIDEO} ({w}x{h} @ {fps:.1f}fps) -> {OUTPUT_VIDEO}")

AQUA = (255, 255, 0)  # 青色
RED = (0, 0, 255)      # 紅色
YELLOW = (0, 255, 255)   # 黃色
FONT = cv2.FONT_HERSHEY_SIMPLEX
last_counts = {}        # 延續上一筆計數，避免無資料的影格閃爍
idx = 0
written = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 重畫偵測框（中心座標 -> 角點）
    _, plist = preds_by_frame.get(idx, (None, []))
    for pred in plist:
        if not isinstance(pred, dict):
            continue
        x, y = pred.get("x"), pred.get("y")
        bw, bh = pred.get("width"), pred.get("height")
        if x is None or bw is None:
            continue
        x1, y1 = int(x - bw / 2), int(y - bh / 2)
        x2, y2 = int(x + bw / 2), int(y + bh / 2)
        cv2.rectangle(frame, (x1, y1), (x2, y2), RED, 1)
        label = f"{pred.get('class', '')}"
        cv2.putText(frame, label, (x1, max(y1 - 4, 10)), FONT, 0.4, RED, 1)

    # 累計計數（無則沿用上一筆）
    if idx in counts_by_frame:
        last_counts = counts_by_frame[idx]
    counts = last_counts

    # 計數警戒線（線段 x=0~160, y=100）
    cv2.line(frame, (COUNT_LINE_X_MIN, COUNT_LINE_Y),
             (COUNT_LINE_X_MAX, COUNT_LINE_Y), AQUA, 1)

    # 左上角總計與各類別
    total = sum(counts.values())
    cv2.putText(frame, f"Total: {total}", (10, 20), FONT, 0.5, YELLOW, 1)
    ty = 38
    for cls_name, count in sorted(counts.items()):
        cv2.putText(frame, f"{cls_name}: {count}", (10, ty), FONT, 0.5, YELLOW, 1)
        ty += 18

    out.write(frame)
    written += 1
    idx += 1

cap.release()
out.release()
conn.close()
print(f"完成：共輸出 {written} 個影格 -> {OUTPUT_VIDEO}")
